# Steam Game Pricing Analysis
## The Importance of Price!
A Pricing Optimization Analysis by Anubad Mahapatra<br>
Available on GitHub

### Imports
For this project we will be using:
* `pandas`
* `numpy`
* `os`
* `datasets`
* `dotenv`
* `scikit-learn`
* `plotly`

### Load the Dataset
The dataset is found on HuggingFace, and is composed of

In [ ]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset
from dotenv import load_dotenv
import re
import warnings
warnings.filterwarnings("ignore")  # Suppress warnings for cleaner output

# Load variables from .env
load_dotenv() 

# Now load the dataset using the token from your environment
print("[1/3] Initializing connection to Hugging Face...")
token = os.getenv("HF_TOKEN") #pull
ds = load_dataset("FronkonGames/steam-games-dataset", token=token)

df = pd.DataFrame(ds["train"])

Loading dataset from Hugging Face...


KeyboardInterrupt: 

### Dataset Exploration

In [ ]:
# 1. Basic Structure
print("\n--- Dataset Info ---")
print(f"Total Rows: {len(df)}")
print(f"Total Columns: {len(df.columns)}")

# 2. Check for missing values (Nulls/NaNs)
print("\n--- Missing Values ---")
missing_data = df.isnull().sum()
print(missing_data[missing_data > 0]) # Only show columns with missing data

# 3. Check for Duplicate entries
# AppID is the unique identifier for a game on Steam
duplicate_count = df.duplicated(subset=['appID']).sum()
print(f"\n--- Duplicates ---")
print(f"Duplicate games found: {duplicate_count}")

# Look at the first 3 rows to confirm it loaded correctly
df.head(3)


--- Dataset Info ---
Total Rows: 124146
Total Columns: 41

--- Missing Values ---
Series([], dtype: int64)

--- Duplicates ---
Duplicate games found: 0


,appID,name,release_date,estimated_owners,peak_ccu,required_age,price,dlc_count,detailed_description,short_description,...,median_playtime_forever,median_playtime_2weeks,developers,publishers,categories,genres,tags,screenshots,movies,packages
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,,,...,0,0,[],[],[],[],[],[https://shared.akamai.steamstatic.com/store_i...,[],[]
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,0,"Springtime, April: when the cherry trees come ...","Spring has come, and our protagonist, Yukinari...",...,8,0,[minori],[MangaGamer],"[Single-player, Steam Trading Cards, Steam Clo...",[Adventure],[],[https://shared.akamai.steamstatic.com/store_i...,[],"[{""title"": ""Buy Supipara - Chapter 1 Spring Ha..."
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,"Immerse yourself in the most beloved, mystical...",Discover an entrancing and spectacular world!,...,0,0,[Somer Games],[8floor],"[Single-player, Family Sharing]",[Casual],[],[https://shared.akamai.steamstatic.com/store_i...,[],"[{""title"": ""Buy Mystery Solitaire The Black Ra..."


### Preprocessing

In [ ]:
print("Starting Preprocessing...")

# 1. Drop Duplicates (if any exist)
if duplicate_count > 0:
    df = df.drop_duplicates(subset=['appID'], keep='first')
    print("Duplicates removed.")

# 2. Function to convert 'estimated_owners' range strings to a numeric midpoint
def parse_owners(owner_str):
    if pd.isna(owner_str) or owner_str == '0 - 0':
        return 0
    # Extract numbers from strings like '20000 - 50000'
    numbers = [int(s) for s in re.findall(r'\d+', str(owner_str))]
    if len(numbers) == 2:
        return sum(numbers) / 2
    return numbers[0] if numbers else 0

# Apply the conversion to create our Demand metric
df['owners_midpoint'] = df['estimated_owners'].apply(parse_owners)

# 3. Date Conversion
# Convert string dates to actual datetime objects so we can filter by year later
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['release_year'] = df['release_date'].dt.year

# 4. Create our Revenue Proxy and filter for "Premium" games
df['est_revenue'] = df['price'] * df['owners_midpoint']

# We filter out Free games (price == 0) because they use a microtransaction 
# business model, which doesn't fit a standard premium price elasticity analysis.
df_premium = df[df['price'] > 0].copy()

print(f"\nPreprocessing Complete!")
print(f"Total Premium Games ready for analysis: {len(df_premium)}")
df_premium[['name', 'price', 'owners_midpoint', 'est_revenue', 'release_year']].head()